In [10]:
import pandas as pd
from autogluon.tabular import TabularPredictor

# 1. 데이터 로드
train = pd.read_csv('./open/train.csv')
test = pd.read_csv('./open/test.csv')

# 2. 피처 엔지니어링 함수 정의
def engineer_features(df):
    # (1) 순서형 데이터 수치화
    mapping = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}
    count_cols = [
        '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
        '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
        '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
    ]
    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(-1)

    # (2) 나이대 수치화 (여기서 '시술 당시 나이'를 사용함)
    age_map = {
        '만18-34세': 26, '만35-37세': 36, '만38-39세': 38.5,
        '만40-42세': 41, '만43-44세': 43.5, '만45-50세': 43.5, '알 수 없음': 43.5
    }
    if '시술 당시 나이' in df.columns:
        df['시술_당시_나이_수치'] = df['시술 당시 나이'].map(age_map)
    
    # (3) 시험관/전환율 관련
    df['난자→배아_전환율'] = df['총 생성 배아 수'].fillna(0) / (df['수집된 신선 난자 수'].fillna(0) + 1)
    df['수정_성공률'] = df['혼합된 난자 수'].fillna(0) / (df['수집된 신선 난자 수'].fillna(0) + 1)
    df['배아_동결_비율'] = df['저장된 배아 수'].fillna(0) / (df['총 생성 배아 수'].fillna(0) + 1)
    df['이식_성공_비율'] = df['이식된 배아 수'].fillna(0) / (df['총 생성 배아 수'].fillna(0) + 1) 
    df['포배기_이식_여부'] = (df['배아 이식 경과일'] == 5.0).astype(int)
    df['분할기_이식_여부'] = (df['배아 이식 경과일'] == 3.0).astype(int)

    # (4) 과거력 및 효율
    df['과거_시술_성공_효율'] = df['총 출산 횟수'] / (df['총 시술 횟수'] + 1)
    df['순수_실패_횟수'] = df['총 시술 횟수'] - df['총 임신 횟수']
    
    return df

# 3. 데이터 적용 (먼저 변수를 생성함)
train_processed = engineer_features(train)
test_processed = engineer_features(test)

# 4. 불필요한 피처 및 중복 피처 삭제 (학습 직전 단계)
drop_features = [
    '시술 유형', 
    '불임 원인 - 자궁경부 문제', 
    '불임 원인 - 정자 농도', 
    '불임 원인 - 정자 형태', 
    '불임 원인 - 정자 운동성',
    '대리모 여부', 
    '난자 해동 경과일', 
    '저장된 신선 난자 수', 
    'DI 출산 횟수',
    'DI 임신 횟수',
    '시술 당시 나이',  # 수치형 변수를 만들었으므로 원본 문자열은 삭제
    'ID'              # ID는 학습에 불필요
]

# 실제로 데이터에 존재하는 컬럼만 골라서 삭제
final_train_data = train_processed.drop(columns=[col for col in drop_features if col in train_processed.columns])
final_test_data = test_processed.drop(columns=[col for col in drop_features if col in test_processed.columns])

print(len(final_train_data.columns),len(final_test_data.columns))


66 65


In [11]:
# 일단은 binary만 추가함/ 2/11 베스트모델 + binary + 기존꺼 drop
def safe_int(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except:
        return None
    
def donor_mix_binary(x):
    v = safe_int(x)
    if v is None:
        return "UNKNOWN"
    if v == 0:
        return "0"
    else:
        return "above 0"


final_train_data["기증자 정자와 혼합된 난자 수_binary"] = final_train_data["기증자 정자와 혼합된 난자 수"].apply(donor_mix_binary)
final_test_data["기증자 정자와 혼합된 난자 수_binary"] = final_test_data["기증자 정자와 혼합된 난자 수"].apply(donor_mix_binary)
final_train_data = final_train_data.drop("기증자 정자와 혼합된 난자 수",axis=1)
final_test_data = final_test_data.drop("기증자 정자와 혼합된 난자 수",axis=1)

def thawed_oocyte_binary(x):
    v = safe_int(x)

    if v is None:
        return "UNKNOWN"
    if v == 0:
        return "0"
    else:
        return "1+"

COL = "해동 난자 수"

final_train_data["해동 난자 수_binary"] = final_train_data["해동 난자 수"].apply(donor_mix_binary)
final_test_data["해동 난자 수_binary"] = final_test_data["해동 난자 수"].apply(donor_mix_binary)
final_train_data = final_train_data.drop("해동 난자 수",axis=1)
final_test_data = final_test_data.drop("해동 난자 수",axis=1)


# 저장된 신선 난자수는 피처 임포텐스에서 너무 낮아서 일단 보류
# def stored_fresh_oocyte_binary(x):
#     v = safe_int(x)

#     if v is None:
#         return "UNKNOWN"
#     if v == 0:
#         return "0"
#     else:
#         return "1+"


# final_train_data["저장된 신선 난자 수_binary"] = final_train_data["저장된 신선 난자 수"].apply(stored_fresh_oocyte_binary)
# final_test_data["저장된 신선 난자 수_binary"] = final_test_data["저장된 신선 난자 수"].apply(stored_fresh_oocyte_binary)
# final_train_data = final_train_data.drop("저장된 신선 난자 수",axis=1)
# final_test_data = final_test_data.drop("저장된 신선 난자 수",axis=1)



print(len(final_train_data.columns),len(final_test_data.columns))

66 65


In [ ]:
# 5. AutoGluon 학습
predictor = TabularPredictor(
    label='임신 성공 여부', 
    eval_metric='roc_auc'
).fit(
    final_train_data,
    presets='best_quality',
    time_limit=7200,
    num_stack_levels=1
)

# 6. 리더보드 확인
leaderboard = predictor.leaderboard(final_train_data, silent=True)
print(leaderboard)

In [3]:
predictions_proba = predictor.predict_proba(final_test_data)

# 임신 성공(1)에 해당하는 확률만 추출
final_preds = predictions_proba.iloc[:, 1] 
 
# 7. 제출 파일 생성
submission = pd.DataFrame({
    'ID': test['ID'],  # 원본 test의 ID를 그대로 사용
    'probability': final_preds
})

# 파일 저장
submission.to_csv('feature_5_2hr_drop_binary_autogluon_submission.csv', index=False, encoding='utf-8')
print("--- 제출 파일 생성 완료 ---")

--- 제출 파일 생성 완료 ---
